# Imports

In [ ]:
import torch

import sys

sys.path.append("..")

import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
from dataclasses import dataclass
from glob import glob

import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import torchvision.transforms.v2 as T
import weightwatcher as ww
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from components.processing.misc import div_255, norm, repeat_rgb_channels, unsqueeze
from models.amber.amber import Amber
from models.amber_no_modality_text.amber_no_modality import AmberNoModality

%matplotlib inline

# Model Load

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "b5hblgur"
model_path = f"../out/{model_id}/ckpt.pt"
model = Amber.from_pretrained(chkpt_path=model_path, device=device, eval_mode=True)

# Weight Watcher

In [ ]:
watcher = ww.WeightWatcher(model=model)
details = watcher.analyze(plot=False)

In [ ]:
details

In [ ]:
set(details["warning"])

In [ ]:
details[details["warning"] == "under-trained"]["longname"].values

In [ ]:
details[details["warning"] == "over-trained"]["longname"].values

In [ ]:
summary = watcher.get_summary(details)
summary

In [ ]:
details.alpha.plot.hist(
    bins=100, title="DINOv2-GPT2 Baseline Model Layer Alphas Distribution"
)
plt.xlabel("weightwatcher layer quality metric alpha")
plt.axvline(x=2.0, color="red")
plt.axvline(x=6.0, color="orange")
plt.show()

# DINOv2 Features

In [ ]:
modality = "cesm"
slice_idx = 0
birads = "benign"

data_path = f"../../data/report_generation_split/{modality}-rg-test.csv"
data = pd.read_csv(data_path)
just_mrs = data[(data["modality"] == modality) & (data["birads"] == birads)]
sample = just_mrs.iloc[slice_idx]
img_path = sample["image_path"]
img_path = os.path.join("..", img_path)
slice = sample["slice"]

img = np.load(img_path)[slice, :, :]

base_transformation = T.Compose(
    [
        T.ToImage(),
        T.Lambda(unsqueeze),
        T.Lambda(repeat_rgb_channels),
        T.Resize(560, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(560),
        T.Lambda(div_255),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)

img = base_transformation(img)
img = img.to(device)

In [ ]:
dinov2_features = model.encoder.forward_features(img)

In [ ]:
dinov2_features.keys()

In [ ]:
numpy_array = dinov2_features["x_norm_clstoken"].mean(0).detach().cpu().numpy()

plt.figure(figsize=(10, 6))
sns.histplot(numpy_array, bins=100, kde=False)  # kde=False if you just want histogram
plt.title(
    f"Mean CLS Token Histogram - 100 bins - {modality} - {birads} - Mean Value: {numpy_array.mean():.2f} - Std Dev: {numpy_array.std():.2f}"
)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.75)
plt.show()

In [ ]:
plt.imshow(
    dinov2_features["x_norm_patchtokens"].detach().cpu().numpy().squeeze(),
    cmap="viridis",
)
plt.title("x_norm_patchtokens")
plt.show()

In [ ]:
numpy_array = (
    dinov2_features["x_norm_patchtokens"].squeeze().mean(0).detach().cpu().numpy()
)

plt.figure(figsize=(10, 6))
sns.histplot(numpy_array, bins=100, kde=False)  # kde=False if you just want histogram
plt.title(
    f"Mean Patch Token Histogram - 100 bins - {modality} - {birads} - Mean Value: {numpy_array.mean():.2f} - Std Dev: {numpy_array.std():.2f}"
)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.75)
plt.show()

# Plot Position Encoders

In [ ]:
tes = model.decoder.transformer.wte
tes = tes.weight

In [ ]:
pes = model.decoder.transformer.wpe
pes = pes.weight

In [ ]:
pes.max(), pes.min(), pes.mean(), pes.std()

In [ ]:
tes.max(), tes.min(), tes.mean(), tes.std()

In [ ]:
plt.hist(pes.flatten().detach().cpu().numpy(), bins=100)
plt.title(
    f"Histogram of Positional Embeddings - 100 bins - Mean Value: {pes.mean().item():.2f} - Std Dev: {pes.std().item():.2f}"
)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.yscale("log")
plt.show()

In [ ]:
plt.hist(tes.flatten().detach().cpu().numpy(), bins=100)
plt.title(
    f"Histogram of Token Embeddings - 100 bins - Mean Value: {tes.mean().item():.2f} - Std Dev: {tes.std().item():.2f}"
)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
pes_correlation_matrix = (pes @ pes.T).detach().cpu().numpy()
pes_correlation_matrix = pes_correlation_matrix / pes_correlation_matrix.max()
(
    pes_correlation_matrix.max(),
    pes_correlation_matrix.min(),
    pes_correlation_matrix.mean(),
    pes_correlation_matrix.std(),
)

In [ ]:
mask = np.eye(pes_correlation_matrix.shape[0], dtype=bool)  # hide diagonal
sns.heatmap(pes_correlation_matrix, mask=mask, robust=True)
plt.show()

# DINOv2 Features Distributions Across Modalities

In [ ]:
base_transformation = T.Compose(
    [
        T.ToImage(),
        T.Lambda(unsqueeze),
        T.Lambda(repeat_rgb_channels),
        T.Resize(560, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(560),
        T.Lambda(div_255),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)

In [ ]:
def sample_samples(modality, n_samples_per_birads=100, random_state=42):
    data_path = f"../../data/report_generation_split/{modality}-rg-test.csv"
    data = pd.read_csv(data_path)
    data = data[(data["modality"] == modality)]

    # group data per "birads" and sample n samples from each group
    result = {}
    grouped_data = data.groupby("birads")

    for name, group in grouped_data:
        sample_n = n_samples_per_birads
        if len(group) < n_samples_per_birads:
            sample_n = len(group)
        sample = group.sample(sample_n, replace=False, random_state=random_state)

        # add the samples to the result such that the key is the birads and the value a list of the samples image_path
        result[name] = sample["image_path"].values
        result[name] = [os.path.join("..", path) for path in result[name]]

    return result

In [ ]:
n_samples_per_birads = 100

mgs = sample_samples("mg", n_samples_per_birads=n_samples_per_birads)
cesms = sample_samples("cesm", n_samples_per_birads=n_samples_per_birads)
us = sample_samples("us", n_samples_per_birads=n_samples_per_birads)
mr = sample_samples("mr", n_samples_per_birads=n_samples_per_birads)

In [ ]:
def prepare_imgs(img_paths):
    imgs = []
    for img_path in img_paths:
        img = np.load(img_path)[0, :, :]
        img = base_transformation(img)
        img = img.to(device)

        dinov2_features = model.encoder.forward_features(img)
        cls = dinov2_features["x_norm_clstoken"]  # ([1, 384])
        patches = dinov2_features["x_norm_patchtokens"]  # ([1, 1600, 384])
        # features = features = torch.cat((cls.unsqueeze(1), patches), dim=1).squeeze(0).mean(0)
        # features = patches.squeeze(0).mean(0)
        imgs.append(cls.squeeze(0).detach().cpu().numpy())
    return imgs

In [ ]:
# for k,v in mgs.items():
#     mgs[k] = prepare_imgs(v)
# for k,v in cesms.items():
#     cesms[k] = prepare_imgs(v)
for k, v in mr.items():
    mr[k] = prepare_imgs(v)
# for k,v in us.items():
#     us[k] = prepare_imgs(v)

In [ ]:
labels = []
imgs = []
# for k,v in mgs.items():
#     imgs.extend(v)
#     labels.extend([f"mg-{k}"] * len(v))

# for k,v in cesms.items():
#     imgs.extend(v)
#     labels.extend([f"cesm-{k}"] * len(v))

for k, v in mr.items():
    imgs.extend(v)
    labels.extend([f"mr-{k}"] * len(v))

# for k,v in us.items():
#     imgs.extend(v)
#     labels.extend([f"us-{k}"] * len(v))

In [ ]:
imgs = np.array(imgs)
labels = np.array(labels)

## T-SNE

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
tsne_result = tsne.fit_transform(imgs)

In [ ]:
tsne_result.shape

In [ ]:
tsne.kl_divergence_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

unique_labels = np.unique(labels)


cmap_name = "tab10" if len(unique_labels) <= 10 else "tab20"
cmap = plt.cm.get_cmap(cmap_name, len(unique_labels))
colors = [cmap(i) for i in range(len(unique_labels))]

plt.figure(figsize=(10, 6))
ax = plt.gca()
for i, lab in enumerate(unique_labels):
    pts = tsne_result[labels == lab, :2]
    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=14,
        alpha=0.5,
        color=colors[i],
        edgecolors="none",
        label=f"{lab} (n={len(pts)})",
    ) 

    ax.scatter(
        pts[:, 0].mean(),
        pts[:, 1].mean(),
        s=70,
        marker="X",
        color=colors[i],
        linewidths=0.6,
        edgecolors="k",
    )
    ax.set_title("t-SNE Separation of Features")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.axhline(0, lw=0.6, c="k", alpha=0.25)
    ax.axvline(0, lw=0.6, c="k", alpha=0.25)
    ax.legend(
        bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, frameon=False
    )

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import cycle


unique_labels = np.unique(labels)


cmap_name = "tab10" if len(unique_labels) <= 10 else "tab20"
cmap = plt.cm.get_cmap(cmap_name, len(unique_labels))
colors = [cmap(i) for i in range(len(unique_labels))]


marker_set = ["o", "s", "^", "D", "P", "X", "v", "<", ">", "*", "h", "H", "p", "8"]
markers = [m for _, m in zip(range(len(unique_labels)), cycle(marker_set))]

plt.figure(figsize=(10, 6))
ax = plt.gca()

for i, lab in enumerate(unique_labels):
    pts = tsne_result[labels == lab, :2]
    m = markers[i]
    c = colors[i]

    sc = ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=22,
        alpha=0.65,
        marker=m,
        facecolors=c,
        edgecolors="k",
        linewidths=0.3,
        label=f"{lab} (n={len(pts)})",
    )

    ax.scatter(
        pts[:, 0].mean(),
        pts[:, 1].mean(),
        s=110,
        marker=m,
        facecolors="none",
        edgecolors=c,
        linewidths=1.4,
    )

ax.set_title("t-SNE Separation of Features")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.axhline(0, lw=0.6, c="k", alpha=0.25)
ax.axvline(0, lw=0.6, c="k", alpha=0.25)


leg = ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0,
    frameon=False,
    scatterpoints=1,
    markerscale=1.4,
    handletextpad=0.6,
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

aggregated_labels = np.array([s.split("-")[0] for s in labels])
groups = np.unique(aggregated_labels)


cmap_name = "tab10" if len(groups) <= 10 else "tab20"
cmap = plt.cm.get_cmap(cmap_name, len(groups))
colors = [cmap(i) for i in range(len(groups))]

plt.figure(figsize=(10, 6))
ax = plt.gca()

for i, g in enumerate(groups):
    pts = tsne_result[aggregated_labels == g, :2]
    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=14,
        alpha=0.5,
        color=colors[i],
        edgecolors="none",
        label=f"{g} (n={len(pts)})",
    )
    # Centroid (remove this block if you want only points)
    ax.scatter(
        pts[:, 0].mean(),
        pts[:, 1].mean(),
        s=70,
        marker="X",
        color=colors[i],
        linewidths=0.6,
        edgecolors="k",
    )

ax.set_title("t-SNE Separation of Features")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.axhline(0, lw=0.6, c="k", alpha=0.25)
ax.axvline(0, lw=0.6, c="k", alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import cycle

aggregated_labels = np.array([s.split("-")[0] for s in labels])
groups = np.unique(aggregated_labels)


cmap_name = "tab10" if len(groups) <= 10 else "tab20"
cmap = plt.cm.get_cmap(cmap_name, len(groups))
colors = [cmap(i) for i in range(len(groups))]


marker_set = ["o", "s", "^", "D", "P", "X", "v", "<", ">", "*", "h", "H", "p", "8"]
markers = [m for _, m in zip(range(len(groups)), cycle(marker_set))]

plt.figure(figsize=(10, 6))
ax = plt.gca()

for i, g in enumerate(groups):
    pts = tsne_result[aggregated_labels == g, :2]
    m = markers[i]
    c = colors[i]

    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=22,
        alpha=0.65,
        marker=m,
        facecolors=c,
        edgecolors="k",
        linewidths=0.3,
        label=f"{g} (n={len(pts)})",
    )

    ax.scatter(
        pts[:, 0].mean(),
        pts[:, 1].mean(),
        s=110,
        marker=m,
        facecolors="none",
        edgecolors=c,
        linewidths=1.4,
    )

ax.set_title("t-SNE Separation of Features")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.axhline(0, lw=0.6, c="k", alpha=0.25)
ax.axvline(0, lw=0.6, c="k", alpha=0.25)


ax.legend(
    bbox_to_anchor=(0, 0.99),
    loc="upper left",
    borderaxespad=0,
    frameon=False,
    scatterpoints=1,
    markerscale=1.4,
    handletextpad=0.6,
)

plt.tight_layout()
plt.show()

## PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(imgs)

In [ ]:
pca_result.shape

In [ ]:
print(
    f"These components cover: {sum(pca.explained_variance_):.2f}% of the total variance in the data"
)

In [ ]:
evr = pca.explained_variance_ratio_
xlab = f"PC1 ({evr[0] * 100:.1f}%)"
ylab = f"PC2 ({evr[1] * 100:.1f}%)"
xlab, ylab

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

unique_labels = np.unique(labels)

cmap_name = "tab10" if len(unique_labels) <= 10 else "tab20"
cmap = plt.cm.get_cmap(cmap_name, len(unique_labels))
colors = [cmap(i) for i in range(len(unique_labels))]

evr = getattr(pca, "explained_variance_ratio_", None)
xlab = (
    f"PC1 ({evr[0] * 100:.1f}%)" if evr is not None and len(evr) > 0 else "Component 1"
)
ylab = (
    f"PC2 ({evr[1] * 100:.1f}%)" if evr is not None and len(evr) > 1 else "Component 2"
)


def confidence_ellipse(X, ax, n_std=1.96, **kwargs):
    if X.shape[0] < 2:
        return
    C = np.cov(X, rowvar=False)
    vals, vecs = np.linalg.eigh(C)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(np.maximum(vals, 1e-12))
    mean = X.mean(axis=0)
    e = Ellipse(xy=mean, width=width, height=height, angle=theta, **kwargs)
    ax.add_patch(e)


plt.figure(figsize=(10, 6))
ax = plt.gca()

for i, lab in enumerate(unique_labels):
    pts = pca_result[labels == lab, :2]
    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=14,
        alpha=0.5,
        color=colors[i],
        edgecolors="none",
        label=f"{lab} (n={len(pts)})",
    )
    ax.scatter(
        pts[:, 0].mean(),
        pts[:, 1].mean(),
        s=70,
        marker="X",
        color=colors[i],
        linewidths=0.6,
        edgecolors="k",
    )
    confidence_ellipse(pts, ax, n_std=1.96, fill=False, lw=1, ec=colors[i], alpha=0.9)

ax.set_title("PCA Separation of Features")
ax.set_xlabel(xlab)
ax.set_ylabel(ylab)
ax.axhline(0, lw=0.6, c="k", alpha=0.25)
ax.axvline(0, lw=0.6, c="k", alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

unique_labels = np.unique(labels)


cmap_name = "tab10" if len(unique_labels) <= 10 else "tab20"
cmap = plt.cm.get_cmap(cmap_name, len(unique_labels))
colors = [cmap(i) for i in range(len(unique_labels))]

evr = getattr(pca, "explained_variance_ratio_", None)
xlab = (
    f"PC1 ({evr[0] * 100:.1f}%)" if evr is not None and len(evr) > 0 else "Component 1"
)
ylab = (
    f"PC2 ({evr[1] * 100:.1f}%)" if evr is not None and len(evr) > 1 else "Component 2"
)

plt.figure(figsize=(10, 6))
ax = plt.gca()

for i, lab in enumerate(unique_labels):
    pts = pca_result[labels == lab, :2]
    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=14,
        alpha=0.5,
        color=colors[i],
        edgecolors="none",
        label=f"{lab} (n={len(pts)})",
    )

ax.set_title("PCA Separation of Features")
ax.set_xlabel(xlab)
ax.set_ylabel(ylab)
ax.axhline(0, lw=0.6, c="k", alpha=0.25)
ax.axvline(0, lw=0.6, c="k", alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Ellipse


def confidence_ellipse(X, ax, n_std=1.96, **kwargs):
    # X: (n,2)
    C = np.cov(X, rowvar=False)
    vals, vecs = np.linalg.eigh(C)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    mean = X.mean(axis=0)
    e = Ellipse(xy=mean, width=width, height=height, angle=theta, **kwargs)
    ax.add_patch(e)
    return e



palette = (
    plt.cm.tab20(np.linspace(0, 1, len(groups)))
    if len(groups) > 10
    else plt.cm.tab10(np.linspace(0, 1, len(groups)))
)

fig, ax = plt.subplots(figsize=(10, 6))
for i, g in enumerate(groups):
    pts = pca_result[aggregated_labels == g, :2]
    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=10,
        alpha=0.35,
        color=palette[i],
        label=f"{g} (n={len(pts)})",
    )
    # centroid
    m = pts.mean(axis=0)
    ax.scatter(*m, s=80, color=palette[i], marker="X", edgecolor="k", linewidths=0.5)
    # ellipse
    confidence_ellipse(pts, ax, n_std=1.96, fill=False, lw=1, ec=palette[i])

ax.set_title("PCA: class centroids and 95% ellipses")
ax.set_xlabel(xlab)
ax.set_ylabel(ylab)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax.axhline(0, c="k", lw=0.4, alpha=0.3)
ax.axvline(0, c="k", lw=0.4, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 6))

aggregated_labels = np.array([s.split("-")[0] for s in labels])
groups = np.unique(aggregated_labels)


palette = (
    plt.cm.tab20(np.linspace(0, 1, len(groups)))
    if len(groups) > 10
    else plt.cm.tab10(np.linspace(0, 1, len(groups)))
)

for i, g in enumerate(groups):
    pts = pca_result[aggregated_labels == g]
    plt.scatter(
        pts[:, 0],
        pts[:, 1],
        s=14,
        alpha=0.5,
        color=palette[i],
        edgecolors="none",
        label=f"{g} (n={len(pts)})",
    )

plt.title("PCA separation of features")
plt.xlabel(xlab)
plt.ylabel(ylab)
plt.axhline(0, lw=0.5, c="k", alpha=0.3)
plt.axvline(0, lw=0.5, c="k", alpha=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

## CKA

In [ ]:
# (1, n_samples, n_features)
stacked_mgs = {}
stacked_cesms = {}
stacked_mr = {}
stacked_us = {}

for k, v in mgs.items():
    stacked_mgs[k] = torch.stack([torch.from_numpy(elem) for elem in mgs[k]])

for k, v in cesms.items():
    stacked_cesms[k] = torch.stack([torch.from_numpy(elem) for elem in cesms[k]])

for k, v in mr.items():
    stacked_mr[k] = torch.stack([torch.from_numpy(elem) for elem in mr[k]])

for k, v in us.items():
    stacked_us[k] = torch.stack([torch.from_numpy(elem) for elem in us[k]])

### Per Modality

In [ ]:
composed_features = {}

for k, v in stacked_mgs.items():
    composed_features[f"mg-{k}"] = v
for k, v in stacked_cesms.items():
    composed_features[f"cesm-{k}"] = v
for k, v in stacked_mr.items():
    composed_features[f"mr-{k}"] = v
for k, v in stacked_us.items():
    composed_features[f"us-{k}"] = v

In [ ]:
# Check variance issues
delta = 1e-6

for k, v in composed_features.items():
    variances = torch.var(v, dim=0)
    num_zeros = torch.sum(variances < delta).item()
    if num_zeros > 0:
        print(
            f"Feature set {k} has {num_zeros} features with variance less than {delta}."
        )

In [ ]:
from components.analyzer.cka import get_cka_matrix

dinov2_cka = torch.zeros(len(composed_features), len(composed_features))

for i, mod_a in tqdm(enumerate(composed_features.keys())):
    for j, mod_b in enumerate(composed_features.keys()):
        cka_matrix = get_cka_matrix(
            composed_features[mod_a].unsqueeze(0), composed_features[mod_b].unsqueeze(0)
        )
        dinov2_cka[i, j] = cka_matrix.item()


# dinov2_cka = dinov2_cka - torch.diag(torch.diag(dinov2_cka)) # set the diagonal to zero for better visualization

plt.imshow(dinov2_cka.detach().cpu().numpy(), cmap="inferno")
plt.colorbar()
plt.title("CKA Matrix")
plt.xticks(
    ticks=np.arange(len(composed_features)),
    labels=list(composed_features.keys()),
    rotation=90,
)
plt.yticks(
    ticks=np.arange(len(composed_features)), labels=list(composed_features.keys())
)
plt.show()

# Models CKA comparison from DINOv2


In [ ]:
from copy import deepcopy

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model_a = Amber.from_pretrained(
    chkpt_path="../out/3h3s05id/ckpt.pt", device=device, eval_mode=True
)
model_b = AmberNoModality.from_pretrained(
    chkpt_path="../out/9fslzrg1/ckpt.pt", device=device, eval_mode=True
)

In [ ]:
base_transformation = T.Compose(
    [
        T.ToImage(),
        T.Lambda(unsqueeze),
        T.Lambda(repeat_rgb_channels),
        T.Resize(560, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(560),
        T.Lambda(div_255),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)


def sample_samples(modality, n_samples_per_birads=100, random_state=42):
    data_path = f"../../data/report_generation_split/{modality}-rg-test.csv"
    data = pd.read_csv(data_path)
    data = data[(data["modality"] == modality)]

    # group data per "birads" and sample n samples from each group
    result = {}
    grouped_data = data.groupby("birads")

    for name, group in grouped_data:
        sample_n = n_samples_per_birads
        if len(group) < n_samples_per_birads:
            sample_n = len(group)
        sample = group.sample(sample_n, replace=False, random_state=random_state)

        # add the samples to the result such that the key is the birads and the value a list of the samples image_path
        result[name] = sample["image_path"].values
        result[name] = [os.path.join("..", path) for path in result[name]]

    return result

In [ ]:
n_samples_per_birads = 100

model_a_mgs = sample_samples("mg", n_samples_per_birads=n_samples_per_birads)
model_a_cesms = sample_samples("cesm", n_samples_per_birads=n_samples_per_birads)
model_a_us = sample_samples("us", n_samples_per_birads=n_samples_per_birads)
model_a_mr = sample_samples("mr", n_samples_per_birads=n_samples_per_birads)

model_b_mgs = deepcopy(model_a_mgs)
model_b_cesms = deepcopy(model_a_cesms)
model_b_us = deepcopy(model_a_us)
model_b_mr = deepcopy(model_a_mr)

In [ ]:
def prepare_imgs_with_model(img_paths, testing_model):
    imgs = []
    for img_path in img_paths:
        img = np.load(img_path)[0, :, :]
        img = base_transformation(img)
        img = img.to(device)

        dinov2_features = testing_model.encoder.forward_features(img)
        cls = dinov2_features["x_norm_clstoken"]  # ([1, 384])
        patches = dinov2_features["x_norm_patchtokens"]  # ([1, 1600, 384])
        # features = features = torch.cat((cls.unsqueeze(1), patches), dim=1).squeeze(0).mean(0)
        # features = patches.squeeze(0).mean(0)
        imgs.append(cls.squeeze(0).detach().cpu().numpy())
    return imgs

In [ ]:
# MODEL A
for k, v in model_a_mgs.items():
    model_a_mgs[k] = prepare_imgs_with_model(v, model_a)
for k, v in model_a_cesms.items():
    model_a_cesms[k] = prepare_imgs_with_model(v, model_a)
for k, v in model_a_mr.items():
    model_a_mr[k] = prepare_imgs_with_model(v, model_a)
for k, v in model_a_us.items():
    model_a_us[k] = prepare_imgs_with_model(v, model_a)

# MODEL B
for k, v in model_b_mgs.items():
    model_b_mgs[k] = prepare_imgs_with_model(v, model_b)
for k, v in model_b_cesms.items():
    model_b_cesms[k] = prepare_imgs_with_model(v, model_b)
for k, v in model_b_mr.items():
    model_b_mr[k] = prepare_imgs_with_model(v, model_b)
for k, v in model_b_us.items():
    model_b_us[k] = prepare_imgs_with_model(v, model_b)

In [ ]:
stacked_model_a_mgs = {}
stacked_model_a_cesms = {}
stacked_model_a_mr = {}
stacked_model_a_us = {}

for k, v in model_a_mgs.items():
    stacked_model_a_mgs[k] = torch.stack(
        [torch.from_numpy(elem) for elem in model_a_mgs[k]]
    )
for k, v in model_a_cesms.items():
    stacked_model_a_cesms[k] = torch.stack(
        [torch.from_numpy(elem) for elem in model_a_cesms[k]]
    )
for k, v in model_a_mr.items():
    stacked_model_a_mr[k] = torch.stack(
        [torch.from_numpy(elem) for elem in model_a_mr[k]]
    )
for k, v in model_a_us.items():
    stacked_model_a_us[k] = torch.stack(
        [torch.from_numpy(elem) for elem in model_a_us[k]]
    )


stacked_model_b_mgs = {}
stacked_model_b_cesms = {}
stacked_model_b_mr = {}
stacked_model_b_us = {}
for k, v in model_b_mgs.items():
    stacked_model_b_mgs[k] = torch.stack(
        [torch.from_numpy(elem) for elem in model_b_mgs[k]]
    )
for k, v in model_b_cesms.items():
    stacked_model_b_cesms[k] = torch.stack(
        [torch.from_numpy(elem) for elem in model_b_cesms[k]]
    )
for k, v in model_b_mr.items():
    stacked_model_b_mr[k] = torch.stack(
        [torch.from_numpy(elem) for elem in model_b_mr[k]]
    )
for k, v in model_b_us.items():
    stacked_model_b_us[k] = torch.stack(
        [torch.from_numpy(elem) for elem in model_b_us[k]]
    )

In [ ]:
from components.analyzer.cka import get_cka_matrix

size = (
    len(stacked_model_a_mgs)
    + len(stacked_model_a_cesms)
    + len(stacked_model_a_mr)
    + len(stacked_model_a_us)
)

dinov2_cka = []
labs = []
for k in stacked_model_a_mgs.keys():
    cka_matrix = get_cka_matrix(
        stacked_model_a_mgs[k].unsqueeze(0), stacked_model_b_mgs[k].unsqueeze(0)
    )
    dinov2_cka.append(cka_matrix.item())
    labs.append(f"mg-{k}")


for k in stacked_model_a_cesms.keys():
    cka_matrix = get_cka_matrix(
        stacked_model_a_cesms[k].unsqueeze(0), stacked_model_b_cesms[k].unsqueeze(0)
    )
    dinov2_cka.append(cka_matrix.item())
    labs.append(f"cesm-{k}")

for k in stacked_model_a_mr.keys():
    cka_matrix = get_cka_matrix(
        stacked_model_a_mr[k].unsqueeze(0), stacked_model_b_mr[k].unsqueeze(0)
    )
    dinov2_cka.append(cka_matrix.item())
    labs.append(f"mr-{k}")

for k in stacked_model_a_us.keys():
    cka_matrix = get_cka_matrix(
        stacked_model_a_us[k].unsqueeze(0), stacked_model_b_us[k].unsqueeze(0)
    )
    dinov2_cka.append(cka_matrix.item())
    labs.append(f"us-{k}")

In [ ]:
y = np.arange(len(labs))

fig = plt.figure(figsize=(7.4, 7.8))
ax = fig.add_subplot(111)

ax.hlines(y=y, xmin=0, xmax=dinov2_cka, linewidth=1.0, alpha=0.6)  # stems
ax.plot(dinov2_cka, y, "o", ms=6)  # markers

# if cis is not None:
#     lowers = np.array([cis[n][0] for n in names])
#     uppers = np.array([cis[n][1] for n in names])
#     ax.hlines(y=y, xmin=lowers, xmax=uppers, linewidth=2.0, alpha=0.35)

ax.set_yticks(y, labs)
ax.set_xlim(0, 1.0)
ax.set_xlabel("Linear CKA")
ax.set_title("Last-layer CKA by category", pad=10)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)

plt.tight_layout()
plt.show()

# Efficiency Analysis

* Compute the FLOPs
* Number of parameters
* CPU vs GPU usage speed

In [ ]:
from thop import profile

model.eval()

dummy_imgs = torch.randn(10, 3, 560, 560).to(device)
dummy_decoder_input_ids = torch.randint(0, 1000, (10, 256)).to(device)
dummy_decoder_mask = torch.ones((10, 256, 256), dtype=torch.long).to(device)
inputs = (dummy_imgs, dummy_decoder_input_ids, None)

with torch.no_grad():
    flops, _ = profile(model, inputs=inputs, verbose=False)

print(f"FLOPs: {flops:,}")

In [ ]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

In [ ]:
bytes_ = pytorch_total_params * 4  # float32 = 4 bytes
mbytes = bytes_ / (1024**2)
gbytes = bytes_ / (1024**3)
print(
    "params:",
    pytorch_total_params,
    "size bytes:",
    bytes_,
    "size MB:",
    mbytes,
    "size GB:",
    gbytes,
)

In [ ]:
b_w = 4  # bytes per weight (float32)
b_a = 4  # bytes per activation (float32)
L = 12  # number of layers
d = 768  # hidden dimension
seq_length = 256  # sequence length
B = 1  # batch size
c = 8  # small constant for attention+MLP workspaces
eps = 0.3  # allocator/framework overhead fraction (GPU: ~0.1–0.3, CPU: ~0.1–0.4)


def compute_model_memory_usage(model_params, b_w, b_a, L, d, T, B, c, eps):
    M_weights = model_params * b_w
    M_transient = c * L * d * T * B * b_a
    M_peak = (M_weights + M_transient) * (1 + eps)
    return M_peak


M_f32_peak_gb = compute_model_memory_usage(
    model_params=pytorch_total_params,
    b_w=4,
    b_a=4,
    L=12,
    d=768,
    T=256,
    B=1,
    c=8,
    eps=0.3,
) / (1024**3)
M_bf16_peak_gb = compute_model_memory_usage(
    model_params=pytorch_total_params,
    b_w=2,
    b_a=2,
    L=12,
    d=768,
    T=256,
    B=1,
    c=8,
    eps=0.3,
) / (1024**3)
print(f"Roughly estimated peak memory usage at fp32: {M_f32_peak_gb:.2f} GB")
print(f"Roughly estimated peak memory usage at bf16: {M_bf16_peak_gb:.2f} GB")

In [ ]:
from time import time

data_path = "../../data/report_generation_split/all-rg-test.csv"
data = pd.read_csv(data_path)
sample = data.sample(n=100, random_state=42, replace=False)
toks_secs = []

for index, row in sample.iterrows():
    img_path = row["image_path"]
    img_path = os.path.join("..", img_path)
    slice = row["slice"]

    img = np.load(img_path)[slice, :, :]

    base_transformation = T.Compose(
        [
            T.ToImage(),
            T.Lambda(unsqueeze),
            T.Lambda(repeat_rgb_channels),
            T.Resize(560, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(560),
            T.Lambda(div_255),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )

    img = base_transformation(img)
    img = img.to(device)

    start_time = time()
    result = model.predict(
        input_images=img, pre_transform=None, plot_attention=False, short_report=False
    )[0]
    end_time = time()
    elapsed_time = end_time - start_time
    toks_sec = result["n_tokens"] / elapsed_time
    toks_secs.append(toks_sec)

print(f"Average tokens per second: {np.mean(toks_secs):.2f} ± {np.std(toks_secs):.2f}")